# 06 - 3-Day Forecast for Shortlisted Models

Train a shortlist of simple models on an 80-day window and save 72-hour (3-day) forecasts to `debug_exports/`.

Default shortlist: `lightgbm`, `random_forest`, `gradient_boosting`, `extra_trees`.
Edit the `SHORTLIST` variable in cell 2 to change models.

In [1]:
import os
import json
import sys
from pathlib import Path

# Force CPU-only for this run
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(str(ROOT / ".env"), override=False)

from src.features.feature_catalog import get_feature_catalog
from src.models.model_configs import MODEL_CONFIGS
from src.models.train import train_model
from src.utils.mongo_client import get_database

ARTIFACTS_DIR = ROOT / "debug_exports"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
SHORTLIST = ["lightgbm", "random_forest", "gradient_boosting", "extra_trees"]
EVAL_WINDOW_DAYS = 80
HORIZON = 72  # 72 hours = 3 days

# Load data from MongoDB
db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))
if "_id" in data.columns:
    data = data.drop(columns=["_id"])
data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

# Prepare window
window_start = data["timestamp"].max() - pd.Timedelta(days=EVAL_WINDOW_DAYS)
window_frame = data.loc[data["timestamp"] >= window_start].sort_values("timestamp").reset_index(drop=True)
if len(window_frame) <= HORIZON:
    raise ValueError("Not enough rows in the selected window to run 3-day forecasts")

# Prepare features
feature_cols = [c for c in get_feature_catalog() if c in window_frame.columns]
feature_frame = window_frame[feature_cols].apply(pd.to_numeric, errors="coerce").ffill().fillna(0.0)
feature_frame = feature_frame.loc[:, feature_frame.nunique(dropna=True) > 1].copy()
if feature_frame.empty or len(feature_frame) <= HORIZON:
    raise ValueError("Unable to prepare feature matrix for forecasting")

target = window_frame["european_aqi"].astype(float)
y = pd.DataFrame([target.iloc[idx: idx + HORIZON].values for idx in range(len(target) - HORIZON)])
x = feature_frame.iloc[: len(y)].copy()

# Train each shortlisted model and save the last-step prediction (next 72 hours)
all_preds = {}
for model_name in SHORTLIST:
    config = next((c for c in MODEL_CONFIGS if c.name == model_name), None)
    if config is None:
        print(f"Skipping unknown model: {model_name}")
        continue
    try:
        model, preds, y_true = train_model(config, x, y, horizon=HORIZON)
        # preds is shape (n_rows, HORIZON); take the last row as the forecast for the next HORIZON hours
        last_pred = preds[-1] if len(preds) > 0 else None
        if last_pred is None:
            print(f"No predictions for {model_name}")
            continue
        df_out = pd.DataFrame({"hour_ahead": list(range(1, HORIZON + 1)), "pred": list(last_pred)})
        out_path = ARTIFACTS_DIR / f"predictions_{model_name}.csv"
        df_out.to_csv(out_path, index=False)
        all_preds[model_name] = df_out
        print(f"Saved predictions for {model_name} to {out_path}")
    except Exception as exc:
        print(f"Model {model_name} failed: {exc}")

# Save combined predictions (wide format)
if all_preds:
    combined = pd.concat([df.set_index('hour_ahead')['pred'].rename(name) for name, df in all_preds.items()], axis=1)
    combined_path = ARTIFACTS_DIR / "predictions_shortlist_combined.csv"
    combined.to_csv(combined_path)
    print(f"Saved combined predictions to {combined_path}")
else:
    print("No predictions produced for shortlist.")

2026-05-30 15:01:00.947648: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780135260.965522  103665 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780135260.970497  103665 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780135260.984416  103665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780135260.984435  103665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780135260.984437  103665 computation_placer.cc:177] computation placer alr

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001186 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13279
[LightGBM] [Info] Number of data points in the train set: 445, number of used features: 186
[LightGBM] [Info] Start training from score 34.698876
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain